# いそがしいひとのための　オントロジーデモ環境自動生成

## つかいかた
このNotebookを任意のワークスペースにインポートし、環境設定を行い全部実行（これだけ！）

## 環境設定
- **WORLSPACE_ID** と **WORKSPACE_NAME** を設定します。
WORKSPACE_ID はURLの/groups/以下の文字列を設定します。
- **LAKEHOUSE_NAME**, **ONTOLOGY_NAME** は[Prefix]_それぞれの名称を入力

## 成果物
- eh_chemical (イベントハウス)
    - chemical_DB
    - eh_chemical
- [Prefix]_ChemicalOntology（オントロジー）
    - [Prefix]_ChemicalOntology_graph_[リソースID] 
    - [Prefix]_ChemicalOntology_lh_[リソースID]
    - [Prefix]_ChemicalOntology_lh_[リソースID]
- [Prefix]_lh_chemical（レイクハウス）
    - [Prefix]_lh_chemical

In [3]:
# 環境設定
WORKSPACE_ID   = "29a1423a-3289-40a2-95f2-e84561970079"    # 👈 Fabric ワークスペースの GUID を入力
WORKSPACE_NAME = "WS_Chemical_Demo"    # 👈 Fabric ワークスペース名を入力
LAKEHOUSE_NAME = "SAKU_lh_chemical"    # 👈 [Prefix]_lh_chemical を入力
ONTOLOGY_NAME  = "SAKU_ChemicalOntology" # [Prefix]_ChemicalOntology

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 5, Finished, Available, Finished, False)

## 依存パッケージのインストール
Step5で実施する内容

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "azure-kusto-data",
                       "azure-kusto-ingest",
                       "azure-identity==1.16.0",
                       "requests>=2.32.3",
                       "--quiet"])

## Step01. Lakehouse の作成
- Workshop 内で実施している Step01 の手順をスクリプトから実施します。

In [ ]:
# Lakehouseの作成
import sempy.fabric as fabric

client = fabric.FabricRestClient()

resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/lakehouses")
resp.raise_for_status()
lakehouses = resp.json().get("value", [])
existing = next((lh for lh in lakehouses if lh["displayName"] == LAKEHOUSE_NAME), None)

if existing:
    LAKEHOUSE_ID = existing["id"]
    print(f"ℹ️  Lakehouse '{LAKEHOUSE_NAME}' は既に存在します — 作成をスキップします。")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")
else:
    resp = client.post(
        f"v1/workspaces/{WORKSPACE_ID}/lakehouses",
        json={"displayName": LAKEHOUSE_NAME},
    )
    resp.raise_for_status()
    LAKEHOUSE_ID = resp.json()["id"]
    print(f"✅ Lakehouse を作成しました: {LAKEHOUSE_NAME}")
    print(f"   LAKEHOUSE_ID = {LAKEHOUSE_ID}")

## Step2. リファレンスデータのダウンロード→アップロード
- https://github.com/miyamam/Chemical-Ontology-Workshop/tree/main/reference_data からリファレンスデータをダウンロードし、Step1で作成したLakehouseにアップロードします。

In [ ]:
import urllib.request, json
from pathlib import Path

BRANCH  = "main"
REPO    = "miyamam/Chemical-Ontology-Workshop"
RAW     = f"https://raw.githubusercontent.com/{REPO}/{BRANCH}"
API     = f"https://api.github.com/repos/{REPO}/git/trees/{BRANCH}?recursive=1"

# ダウンロード対象フォルダ → ./builtin/ にミラーリング
# (repo_prefix, local_sub) — repo 側パス → builtin 配下のローカルパス
FOLDERS = [
    ("Scripts",                  ""),                       # builtin/ 直下に展開
    ("reference_data",  "reference_data"),
]

BUILTIN = Path("./builtin")
BUILTIN.mkdir(exist_ok=True)

def _download(url, dest):
    dest.parent.mkdir(parents=True, exist_ok=True)
    req = urllib.request.Request(url, headers={"User-Agent": "FabricNotebook/1.0"})
    with urllib.request.urlopen(req) as r:
        dest.write_bytes(r.read())

print(f"ブランチ '{BRANCH}' のファイルツリーを取得中...")
req = urllib.request.Request(API, headers={"User-Agent": "FabricNotebook/1.0"})
with urllib.request.urlopen(req) as r:
    tree = json.loads(r.read())["tree"]
print(f"  リポジトリ内 {len(tree)} エントリ\n")

for repo_prefix, local_sub in FOLDERS:
    blobs = [e for e in tree if e["type"] == "blob" and e["path"].startswith(repo_prefix + "/")]
    label = local_sub or repo_prefix
    print(f"📥 '{repo_prefix}/' から {len(blobs)} ファイルをダウンロード中...")
    dest_root = BUILTIN / local_sub if local_sub else BUILTIN
    for entry in blobs:
        rel      = entry["path"][len(repo_prefix) + 1:]
        dest     = dest_root / rel
        url      = f"{RAW}/{entry['path']}"
        _download(url, dest)
        print(f"   ✅ {rel}")
    print()

print(f"✅ 全アセットのダウンロード完了 → {BUILTIN.resolve()}")

In [ ]:
# リファレンスデータのアップロード
import os
from pathlib import Path

LOCAL_TMP      = Path("./builtin/reference_data")
ONELAKE_DEST   = (
    f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com"
    f"/{LAKEHOUSE_ID}/Files"
)

jsonl_files = sorted(LOCAL_TMP.glob("*.jsonl"))
if not jsonl_files:
    raise RuntimeError("JSONL ファイルが見つかりません。上記の出力にエラーがないか確認してください。")

print(f"\n{len(jsonl_files)} ファイルを Lakehouse にアップロード中...")
for f in jsonl_files:
    dest = f"{ONELAKE_DEST}/{f.name}"
    mssparkutils.fs.put(dest, f.read_text(encoding="utf-8"), True)
    print(f"   ✅ {f.name:<35} {f.stat().st_size / 1024:>7.1f} KB")

print(f"\n🎉 {len(jsonl_files)} ファイルを Lakehouse に書き込みました。")
print(f"   Lakehouse : {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")
print( "   次のステップ: Step 3 でこれらのファイルから Delta テーブルを作成します。")

# Step3.Deltaテーブルの作成

## スキーマ定義

In [ ]:
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, FloatType, DoubleType,
    BooleanType, DateType, TimestampType, ArrayType, LongType
)

SCHEMAS = {
    "production_lines": StructType([
        StructField("production_line_id", StringType(), False),
        StructField("name", StringType(), False),
        StructField("plant", StringType(), False),
        StructField("process_type", StringType(), False),
        StructField("capacity_tons_per_day", DoubleType(), True),
        StructField("status", StringType(), False),
    ]),
    "equipment": StructType([
        StructField("equipment_id", StringType(), False),
        StructField("equipment_number", StringType(), False),
        StructField("name", StringType(), False),
        StructField("equipment_type", StringType(), False),
        StructField("production_line_id", StringType(), False),
        StructField("installation_year", IntegerType(), True),
        StructField("manufacturer", StringType(), True),
        StructField("status", StringType(), False),
        StructField("last_maintenance_date", StringType(), True),
    ]),
    "sensors": StructType([
        StructField("sensor_id", StringType(), False),
        StructField("tag_name", StringType(), False),
        StructField("measurement_type", StringType(), False),
        StructField("unit", StringType(), False),
        StructField("equipment_id", StringType(), False),
        StructField("normal_min", DoubleType(), True),
        StructField("normal_max", DoubleType(), True),
        StructField("alarm_low", DoubleType(), True),
        StructField("alarm_high", DoubleType(), True),
        StructField("status", StringType(), False),
    ]),
    "products": StructType([
        StructField("product_id", StringType(), False),
        StructField("name", StringType(), False),
        StructField("grade", StringType(), False),
        StructField("spec_lower", DoubleType(), True),
        StructField("spec_upper", DoubleType(), True),
        StructField("unit", StringType(), True),
        StructField("category", StringType(), False),
    ]),
    "process_orders": StructType([
        StructField("process_order_id", StringType(), False),
        StructField("batch_number", StringType(), False),
        StructField("product_id", StringType(), False),
        StructField("production_line_id", StringType(), False),
        StructField("target_quantity", DoubleType(), True),
        StructField("actual_quantity", DoubleType(), True),
        StructField("start_time", StringType(), True),
        StructField("end_time", StringType(), True),
        StructField("status", StringType(), False),
    ]),
    "operation_phases": StructType([
        StructField("operation_phase_id", StringType(), False),
        StructField("process_order_id", StringType(), False),
        StructField("phase_name", StringType(), False),
        StructField("sequence_number", IntegerType(), False),
        StructField("set_temperature", DoubleType(), True),
        StructField("set_pressure", DoubleType(), True),
        StructField("actual_temperature", DoubleType(), True),
        StructField("actual_pressure", DoubleType(), True),
        StructField("primary_sensor_id", StringType(), True),
        StructField("start_time", StringType(), True),
        StructField("end_time", StringType(), True),
        StructField("status", StringType(), False),
    ]),
    "process_deviations": StructType([
        StructField("process_deviation_id", StringType(), False),
        StructField("sensor_id", StringType(), False),
        StructField("operation_phase_id", StringType(), False),
        StructField("deviation_type", StringType(), False),
        StructField("detection_time", StringType(), True),
        StructField("deviation_amount", DoubleType(), True),
        StructField("threshold_value", DoubleType(), True),
        StructField("actual_value", DoubleType(), True),
        StructField("handling_status", StringType(), False),
    ]),
    "quality_results": StructType([
        StructField("quality_result_id", StringType(), False),
        StructField("lot_number", StringType(), False),
        StructField("process_order_id", StringType(), False),
        StructField("product_id", StringType(), False),
        StructField("process_deviation_id", StringType(), True),
        StructField("inspection_item", StringType(), False),
        StructField("measured_value", DoubleType(), True),
        StructField("spec_lower", DoubleType(), True),
        StructField("spec_upper", DoubleType(), True),
        StructField("pass_fail", StringType(), False),
        StructField("inspection_date", StringType(), True),
    ]),
    "failure_events": StructType([
        StructField("failure_event_id", StringType(), False),
        StructField("equipment_id", StringType(), False),
        StructField("process_deviation_id", StringType(), True),
        StructField("occurrence_time", StringType(), True),
        StructField("failure_mode", StringType(), False),
        StructField("impact_scope", StringType(), False),
        StructField("downtime_hours", DoubleType(), True),
        StructField("severity", StringType(), False),
        StructField("resolution_status", StringType(), False),
    ]),
    "root_causes": StructType([
        StructField("root_cause_id", StringType(), False),
        StructField("failure_event_id", StringType(), False),
        StructField("equipment_id", StringType(), False),
        StructField("operation_phase_id", StringType(), True),
        StructField("cause_classification", StringType(), False),
        StructField("description", StringType(), True),
        StructField("corrective_action", StringType(), True),
        StructField("preventive_action", StringType(), True),
        StructField("analysis_date", StringType(), True),
    ]),
}

# Primary key column for each table
PRIMARY_KEYS = {
    "production_lines": "production_line_id",
    "equipment": "equipment_id",
    "sensors": "sensor_id",
    "products": "product_id",
    "process_orders": "process_order_id",
    "operation_phases": "operation_phase_id",
    "process_deviations": "process_deviation_id",
    "quality_results": "quality_result_id",
    "failure_events": "failure_event_id",
    "root_causes": "root_cause_id",
}

## JSONLファイルをLakehouseテーブルに読み込む

In [ ]:
REFERENCE_DATA_PATH = f"Files"

# Order matters: load tables with no FK dependencies first
LOAD_ORDER = [
    "production_lines",
    "equipment",
    "sensors",
    "products",
    "process_orders",
    "operation_phases",
    "process_deviations",
    "quality_results",
    "failure_events",
    "root_causes",
]

results = []

for table_name in LOAD_ORDER:
    file_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/{REFERENCE_DATA_PATH}/{table_name}.jsonl"
    schema = SCHEMAS[table_name]
    pk = PRIMARY_KEYS[table_name]

    try:
        # Read JSONL (JSON Lines format = one JSON object per line)
        df = spark.read.schema(schema).json(file_path)

        # Verify no null primary keys
        null_pk_count = df.filter(df[pk].isNull()).count()
        if null_pk_count > 0:
            print(f"⚠ WARNING: {table_name} has {null_pk_count} null primary keys ({pk})")

        # Verify primary key uniqueness
        total = df.count()
        distinct_pk = df.select(pk).distinct().count()
        if total != distinct_pk:
            print(f"⚠ WARNING: {table_name} has {total - distinct_pk} duplicate primary keys ({pk})")

        # Write as Delta table (overwrite to support re-runs)
        df.write.format("delta").mode("overwrite").saveAsTable(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{table_name}")

        results.append({"table": table_name, "rows": total, "pk": pk, "status": "✓ loaded"})
        print(f"  ✓ {table_name}: {total} rows loaded (PK: {pk})")

    except Exception as e:
        results.append({"table": table_name, "rows": 0, "pk": pk, "status": f"✗ {str(e)[:80]}"})

In [ ]:
# summary display
from pyspark.sql import Row

summary_df = spark.createDataFrame([Row(**r) for r in results])
display(summary_df)

## 検証：行数とサンプルデータ
このステップはスキップしても問題ありません。

In [ ]:
print("=" * 60)
print("TABLE ROW COUNTS")
print("=" * 60)
for table_name in LOAD_ORDER:
     try:
         count = spark.table(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{table_name}").count() #Lakehouse スキーマを使用しない場合
         print(f"  {table_name:<25} {count:>6} rows")
     except Exception as e:
         print(f"  {table_name:<25} ERROR: {e}")

In [ ]:
# Display sample records from key tables
for table_name in ["production_lines", "equipment", "sensors", "process_orders"]:
     print(f"\n{'=' * 60}")
     print(f"SAMPLE: {table_name} (first 3 rows)")
     print("=" * 60)
     display(spark.table(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{table_name}").limit(3)) #Lakehouseスキーマを使用しない場合

# Step4. Eventhouse & KQL Database の作成

In [ ]:
import importlib.util
from pathlib import Path

eh_script = Path("./builtin/create_chemical_eventhouse.py")

if eh_script.exists():
    spec = importlib.util.spec_from_file_location(
        "create_chemical_eventhouse", str(eh_script)
    )
    eh = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(eh)

    client = eh._get_client()

    print("[1/3] Eventhouse...")
    EVENTHOUSE_ID = eh.ensure_eventhouse(WORKSPACE_ID, client)

    print("\n[2/3] KQL Database...")
    KQL_DB_ID = eh.ensure_kql_database(WORKSPACE_ID, EVENTHOUSE_ID, client)

    print(f"\n✅ Eventhouse  : {eh.EVENTHOUSE_NAME} ({EVENTHOUSE_ID})")
    print(f"   KQL Database : {eh.KQL_DB_NAME} ({KQL_DB_ID})")

    print("\n[3/3] KQL Tables...")
    created = eh.create_kql_tables(WORKSPACE_ID, EVENTHOUSE_ID, eh.KQL_DB_NAME, client)
    print(f"\n✅ テーブル準備完了: {', '.join(created) if created else '(なし)'}")
else:
    print("⚠️ create_chemical_eventhouse.py が見つかりません。")
    print("   Eventhouse と KQL Database を Fabric ポータルから手動で作成してください。")
    print("\n   手動作成手順:")
    print("   1. ワークスペースで [+ 新しいアイテム] → [Eventhouse] を選択")
    print("   2. 名前: [Prefix]_eh_chemical")
    print("   3. KQL Database: chemical_db を作成")
    print("   4. 以下の 5 つの KQL テーブルを作成:")
    print("      - SensorReadingEvent")
    print("      - ProcessAlarmEvent")
    print("      - EquipmentStatusEvent")
    print("      - BatchPhaseTransitionEvent")
    print("      - QualityInspectionEvent")
    print("\n   EVENTHOUSE_ID と KQL_DB_ID を手動で設定してください:")
    EVENTHOUSE_ID = ""  # 👈 手動作成した場合はここに入力
    KQL_DB_ID = ""

## KQL テーブルの検証

In [ ]:
import requests

if EVENTHOUSE_ID:
    # Eventhouse クエリ URI を解決
    resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/eventhouses/{EVENTHOUSE_ID}")
    resp.raise_for_status()
    KUSTO_URI = resp.json()["properties"]["queryServiceUri"]
    KUSTO_DATABASE = "chemical_db"

    kusto_token = notebookutils.credentials.getToken("https://kusto.kusto.windows.net")
    headers = {
        "Authorization": f"Bearer {kusto_token}",
        "Content-Type": "application/json",
        "Accept": "application/json",
    }
    mgmt_url = f"{KUSTO_URI}/v1/rest/mgmt"

    verify_stmt = ".show tables | project TableName"
    body = {"db": KUSTO_DATABASE, "csl": verify_stmt, "properties": {}}
    r = requests.post(mgmt_url, headers=headers, json=body, timeout=60)
    r.raise_for_status()

    rows = r.json().get("Tables", [{}])[0].get("Rows", [])
    existing_tables = sorted(row[0] for row in rows)

    expected = {
        "SensorReadingEvent", "ProcessAlarmEvent", "EquipmentStatusEvent",
        "BatchPhaseTransitionEvent", "QualityInspectionEvent",
    }
    print(f"'{KUSTO_DATABASE}' 内のテーブル:\n")
    for name in existing_tables:
        status = "✅" if name in expected else "  "
        print(f"  {status} {name}")

    missing = expected - set(existing_tables)
    if missing:
        print(f"\n⚠️  不足テーブル: {', '.join(sorted(missing))}")
    else:
        print(f"\n✅ 全 {len(expected)} イベントテーブルを確認しました。")
else:
    print("ℹ️  EVENTHOUSE_ID が未設定のため、KQL テーブルの検証をスキップします。")

# Step5.イベントストリームの生成

## 依存パッケージのインストール
冒頭で行っているので、コメントアウト

import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install",
                       "azure-kusto-data",
                       "azure-kusto-ingest",
                       "azure-identity==1.16.0",
                       "requests>=2.32.3",
                       "--quiet"])

## シミュレーション設定

In [24]:
import json
import uuid
import random
import time
from datetime import datetime, timedelta

# シミュレーションパラメータ
SIMULATION_DURATION_MINUTES = 1      # シミュレーション実行時間（分）
SENSOR_READING_INTERVAL_SECONDS = 10 # センサー計測間隔（秒）
SCENARIO_INJECTION_ENABLED = True    # デモシナリオの注入
BATCH_SIZE = 100                     # Kusto 取り込みバッチサイズ
PRINT_EVENTS = True                  # イベントをコンソールに出力

random.seed(None)

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 26, Finished, Available, Finished, False)

## Eventhouse（KQL Database）への接続

In [25]:
from azure.kusto.data import KustoClient, KustoConnectionStringBuilder, DataFormat
from azure.kusto.ingest import QueuedIngestClient, IngestionProperties
import io
import pandas as pd

KUSTO_TOKEN_SCOPE = "https://kusto.kusto.windows.net"

def get_fabric_token():
    """Fabric 組み込み認証プロバイダーからアクセストークンを取得"""
    return mssparkutils.credentials.getToken(KUSTO_TOKEN_SCOPE)

kcsb = KustoConnectionStringBuilder.with_token_provider(
    KUSTO_URI, get_fabric_token
)
kusto_client = KustoClient(kcsb)

ingest_uri = KUSTO_URI.replace("https://", "https://ingest-")
ingest_kcsb = KustoConnectionStringBuilder.with_token_provider(
    ingest_uri, get_fabric_token
)
ingest_client = QueuedIngestClient(ingest_kcsb)

result = kusto_client.execute(KUSTO_DATABASE, ".show database schema | count")
for row in result.primary_results[0]:
    print(f"✓ Eventhouse に接続しました: {KUSTO_URI}")
    print(f"  Database: {KUSTO_DATABASE}")

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 27, Finished, Available, Finished, False)

✓ Eventhouse に接続しました: https://trd-4k5wq836f2nequ7k1u.z9.kusto.fabric.microsoft.com
  Database: chemical_db


## 取り込みヘルパー

In [26]:
# 各テーブルのカラム順序（.create-merge-table スキーマと一致させること）
TABLE_COLUMNS = {
    "SensorReadingEvent": [
        "event_id", "event_type", "timestamp", "source",
        "sensor_id", "equipment_id", "production_line_id", "process_order_id",
        "tag_name", "measurement_type", "value", "unit",
        "normal_min", "normal_max", "alarm_low", "alarm_high",
        "is_within_normal",
    ],
    "ProcessAlarmEvent": [
        "event_id", "event_type", "timestamp", "source",
        "sensor_id", "equipment_id", "production_line_id", "process_order_id",
        "tag_name", "alarm_type", "severity",
        "threshold_value", "actual_value", "deviation_amount",
        "action_taken",
    ],
    "EquipmentStatusEvent": [
        "event_id", "event_type", "timestamp", "source",
        "equipment_id", "production_line_id",
        "equipment_name", "equipment_type",
        "previous_status", "new_status", "reason",
    ],
    "BatchPhaseTransitionEvent": [
        "event_id", "event_type", "timestamp", "source",
        "process_order_id", "batch_number", "product_id", "production_line_id",
        "previous_phase", "new_phase", "sequence_number",
        "set_temperature", "set_pressure",
        "actual_temperature", "actual_pressure",
    ],
    "QualityInspectionEvent": [
        "event_id", "event_type", "timestamp", "source",
        "process_order_id", "batch_number", "product_id",
        "inspection_item", "measured_value",
        "spec_lower", "spec_upper", "pass_fail",
        "lot_number",
    ],
}

_event_buffers = {table: [] for table in TABLE_COLUMNS}
_ingest_counts = {table: 0 for table in TABLE_COLUMNS}

def flatten_event(event):
    row = {
        "event_id": event["event_id"],
        "event_type": event["event_type"],
        "timestamp": event["timestamp"],
        "source": event["source"],
    }
    row.update(event["body"])
    return row

def buffer_event(event):
    table = event["event_type"]
    row = flatten_event(event)
    _event_buffers[table].append(row)
    if len(_event_buffers[table]) >= BATCH_SIZE:
        flush_buffer(table)

def flush_buffer(table):
    rows = _event_buffers[table]
    if not rows:
        return
    columns = TABLE_COLUMNS[table]
    df = pd.DataFrame(rows, columns=columns)
    props = IngestionProperties(
        database=KUSTO_DATABASE,
        table=table,
        data_format=DataFormat.CSV,
    )
    csv_buffer = io.StringIO()
    df.to_csv(csv_buffer, index=False, header=False)
    csv_buffer.seek(0)
    csv_bytes = io.BytesIO(csv_buffer.getvalue().encode("utf-8"))
    ingest_client.ingest_from_stream(csv_bytes, ingestion_properties=props)
    _ingest_counts[table] += len(rows)
    _event_buffers[table] = []

def flush_all_buffers():
    for table in TABLE_COLUMNS:
        flush_buffer(table)

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 28, Finished, Available, Finished, False)

## Lakehouseからリファレンスデータを読み込み

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()
lakehouse_path = f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}/Tables" #スキーマが無い場合

production_lines_df = spark.read.format("delta").load(f"{lakehouse_path}/production_lines").toPandas()
equipment_df        = spark.read.format("delta").load(f"{lakehouse_path}/equipment").toPandas()
sensors_df          = spark.read.format("delta").load(f"{lakehouse_path}/sensors").toPandas()
products_df         = spark.read.format("delta").load(f"{lakehouse_path}/products").toPandas()
process_orders_df   = spark.read.format("delta").load(f"{lakehouse_path}/process_orders").toPandas()
operation_phases_df = spark.read.format("delta").load(f"{lakehouse_path}/operation_phases").toPandas()

# ルックアップ辞書の構築
production_lines = {row["production_line_id"]: row.to_dict() for _, row in production_lines_df.iterrows()}
equipment = {row["equipment_id"]: row.to_dict() for _, row in equipment_df.iterrows()}
sensors = {row["sensor_id"]: row.to_dict() for _, row in sensors_df.iterrows()}
products = {row["product_id"]: row.to_dict() for _, row in products_df.iterrows()}
process_orders = {row["process_order_id"]: row.to_dict() for _, row in process_orders_df.iterrows()}

# センサー → 設備 → 製造ラインの関連付け
sensor_to_equipment = {s["sensor_id"]: s["equipment_id"] for s in sensors.values()}
equipment_to_line = {e["equipment_id"]: e["production_line_id"] for e in equipment.values()}

# 実行中のプロセスオーダーを取得
active_orders = process_orders_df[process_orders_df["status"] == "running"].to_dict("records")

# 実行中オーダーに紐づくフェーズを取得
active_order_ids = {o["process_order_id"] for o in active_orders}
active_phases = operation_phases_df[
    operation_phases_df["process_order_id"].isin(active_order_ids)
].to_dict("records")

# 稼働中の設備のセンサーリスト
running_equipment_ids = {e["equipment_id"] for e in equipment.values() if e["status"] == "running"}
active_sensors = [s for s in sensors.values() if s["equipment_id"] in running_equipment_ids]

print(f"リファレンスデータを読み込みました:")
print(f"  製造ライン: {len(production_lines)}")
print(f"  設備: {len(equipment)}（稼働中: {len(running_equipment_ids)}）")
print(f"  センサー: {len(sensors)}（アクティブ: {len(active_sensors)}）")
print(f"  プロセスオーダー: {len(process_orders)}（実行中: {len(active_orders)}）")
print(f"  オペレーションフェーズ: {len(active_phases)}")

## ヘルパー関数

In [29]:
def new_event_id():
    return str(uuid.uuid4())

def make_envelope(event_type, source, body):
    """イベント本体を標準エンベロープでラップ"""
    return {
        "event_id": new_event_id(),
        "event_type": event_type,
        "timestamp": datetime.utcnow().isoformat() + "Z",
        "source": source,
        "body": body,
    }

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 31, Finished, Available, Finished, False)

## アラーム種別リファレンス

In [30]:
ALARM_TYPES = [
    {"alarm_type": "temperature_high",  "description": "温度上限超過",   "severity": "critical"},
    {"alarm_type": "temperature_low",   "description": "温度下限逸脱",   "severity": "warning"},
    {"alarm_type": "pressure_high",     "description": "圧力上限超過",   "severity": "critical"},
    {"alarm_type": "pressure_low",      "description": "圧力下限逸脱",   "severity": "warning"},
    {"alarm_type": "vibration_high",    "description": "振動値異常",     "severity": "warning"},
    {"alarm_type": "pH_out_of_range",   "description": "pH 規格外",      "severity": "critical"},
    {"alarm_type": "flow_anomaly",      "description": "流量異常",       "severity": "warning"},
    {"alarm_type": "level_alarm",       "description": "液位アラーム",   "severity": "info"},
]

EQUIPMENT_FAULT_REASONS = [
    "ベアリング過熱",
    "モーター過電流",
    "シール漏洩検知",
    "振動異常値超過",
    "インバーター故障",
    "冷却水流量低下",
    "計装エア圧力低下",
]

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 32, Finished, Available, Finished, False)

## プラント状態トラッカー

In [31]:
class PlantSimState:
    """化学プラントのリアルタイムシミュレーション状態を管理"""

    def __init__(self, sensor, equip, line, order=None, phase=None):
        self.sensor = sensor
        self.equipment = equip
        self.production_line = line
        self.order = order
        self.phase = phase

        # センサー現在値（正規範囲内でランダム初期化）
        nmin = float(sensor.get("normal_min", 0))
        nmax = float(sensor.get("normal_max", 100))
        self.current_value = random.uniform(nmin, nmax)

        # シナリオ注入フラグ
        self.temp_excursion_injected = False
        self.equipment_trip_injected = False
        self.pressure_spike_injected = False

    def advance(self):
        """センサー値を小幅に変動させる"""
        nmin = float(self.sensor.get("normal_min", 0))
        nmax = float(self.sensor.get("normal_max", 100))
        spread = (nmax - nmin) * 0.05
        self.current_value += random.uniform(-spread, spread)
        # 正規範囲に緩やかに引き戻す
        midpoint = (nmin + nmax) / 2
        self.current_value += (midpoint - self.current_value) * 0.02

    @property
    def is_within_normal(self):
        nmin = float(self.sensor.get("normal_min", 0))
        nmax = float(self.sensor.get("normal_max", 100))
        return nmin <= self.current_value <= nmax

    @property
    def is_alarm(self):
        alow = float(self.sensor.get("alarm_low", -999999))
        ahigh = float(self.sensor.get("alarm_high", 999999))
        return self.current_value < alow or self.current_value > ahigh

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 33, Finished, Available, Finished, False)

## イベントジェネレーター

In [32]:
def generate_sensor_reading(state):
    """SensorReadingEvent を生成"""
    s = state.sensor
    return make_envelope("SensorReadingEvent", "dcs-historian", {
        "sensor_id": s["sensor_id"],
        "equipment_id": s["equipment_id"],
        "production_line_id": state.production_line["production_line_id"],
        "process_order_id": state.order["process_order_id"] if state.order else None,
        "tag_name": s["tag_name"],
        "measurement_type": s["measurement_type"],
        "value": round(state.current_value, 3),
        "unit": s["unit"],
        "normal_min": float(s.get("normal_min", 0)),
        "normal_max": float(s.get("normal_max", 100)),
        "alarm_low": float(s.get("alarm_low", -999)),
        "alarm_high": float(s.get("alarm_high", 999)),
        "is_within_normal": state.is_within_normal,
    })


def generate_process_alarm(state, alarm_type_info):
    """ProcessAlarmEvent を生成"""
    s = state.sensor
    threshold = float(s.get("alarm_high", 999)) if "high" in alarm_type_info["alarm_type"] else float(s.get("alarm_low", -999))
    return make_envelope("ProcessAlarmEvent", "alarm-management", {
        "sensor_id": s["sensor_id"],
        "equipment_id": s["equipment_id"],
        "production_line_id": state.production_line["production_line_id"],
        "process_order_id": state.order["process_order_id"] if state.order else None,
        "tag_name": s["tag_name"],
        "alarm_type": alarm_type_info["alarm_type"],
        "severity": alarm_type_info["severity"],
        "threshold_value": threshold,
        "actual_value": round(state.current_value, 3),
        "deviation_amount": round(abs(state.current_value - threshold), 3),
        "action_taken": random.choice(["operator_notified", "auto_adjusted", "escalated"]),
    })


def generate_equipment_status(equip, line, prev_status, new_status, reason):
    """EquipmentStatusEvent を生成"""
    return make_envelope("EquipmentStatusEvent", "plc-gateway", {
        "equipment_id": equip["equipment_id"],
        "production_line_id": line["production_line_id"],
        "equipment_name": equip["name"],
        "equipment_type": equip["equipment_type"],
        "previous_status": prev_status,
        "new_status": new_status,
        "reason": reason,
    })


def generate_batch_phase_transition(order, prev_phase, new_phase, seq, line_id):
    """BatchPhaseTransitionEvent を生成"""
    return make_envelope("BatchPhaseTransitionEvent", "mes-system", {
        "process_order_id": order["process_order_id"],
        "batch_number": order["batch_number"],
        "product_id": order["product_id"],
        "production_line_id": line_id,
        "previous_phase": prev_phase,
        "new_phase": new_phase,
        "sequence_number": seq,
        "set_temperature": round(random.uniform(60, 180), 1),
        "set_pressure": round(random.uniform(0.05, 3.0), 3),
        "actual_temperature": round(random.uniform(58, 182), 1),
        "actual_pressure": round(random.uniform(0.04, 3.1), 3),
    })


def generate_quality_inspection(order, product):
    """QualityInspectionEvent を生成"""
    spec_lower = float(product.get("spec_lower", 0))
    spec_upper = float(product.get("spec_upper", 100))
    # 90% の確率で合格範囲内
    if random.random() < 0.9:
        measured = random.uniform(spec_lower, spec_upper)
        pass_fail = "pass"
    else:
        offset = (spec_upper - spec_lower) * random.uniform(0.01, 0.1)
        measured = spec_upper + offset if random.random() < 0.5 else spec_lower - offset
        pass_fail = "fail"
    inspection_items = ["purity", "viscosity", "moisture_content", "particle_size", "density", "pH"]
    return make_envelope("QualityInspectionEvent", "lims-system", {
        "process_order_id": order["process_order_id"],
        "batch_number": order["batch_number"],
        "product_id": order["product_id"],
        "inspection_item": random.choice(inspection_items),
        "measured_value": round(measured, 4),
        "spec_lower": spec_lower,
        "spec_upper": spec_upper,
        "pass_fail": pass_fail,
        "lot_number": f"L-{datetime.utcnow().strftime('%Y')}-{random.randint(1000,9999):04d}",
    })

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 34, Finished, Available, Finished, False)

## デモシナリオ注入

In [33]:
def check_and_inject_scenarios(states, events):
    """
    デモシナリオの条件を確認し、必要に応じてイベントを注入する。
    """
    for state in states:
        s = state.sensor
        mtype = s.get("measurement_type", "")

        # --- シナリオ 1: 温度逸脱 ---
        if (not state.temp_excursion_injected
                and mtype == "temperature"
                and random.random() < 0.08):
            state.temp_excursion_injected = True
            alarm_high = float(s.get("alarm_high", 999))
            state.current_value = alarm_high + random.uniform(5, 20)
            alarm = next((a for a in ALARM_TYPES if a["alarm_type"] == "temperature_high"), ALARM_TYPES[0])
            print(f"\n  🚨 シナリオ: 温度逸脱 — {s['tag_name']} ({state.equipment['name']})")
            print(f"     実測値: {state.current_value:.1f}{s['unit']}, アラーム閾値: {alarm_high}{s['unit']}")
            events.append(generate_process_alarm(state, alarm))

        # --- シナリオ 2: 設備トリップ ---
        if (not state.equipment_trip_injected
                and state.equipment.get("equipment_type") in ("reactor", "compressor", "pump")
                and random.random() < 0.05):
            state.equipment_trip_injected = True
            reason = random.choice(EQUIPMENT_FAULT_REASONS)
            line = state.production_line
            print(f"\n  🚨 シナリオ: 設備トリップ — {state.equipment['name']} ({state.equipment['equipment_number']})")
            print(f"     原因: {reason}")
            print(f"     製造ライン: {line['name']}")
            events.append(generate_equipment_status(
                state.equipment, line, "running", "fault", reason
            ))

        # --- シナリオ 3: 圧力スパイク ---
        if (not state.pressure_spike_injected
                and mtype == "pressure"
                and random.random() < 0.06):
            state.pressure_spike_injected = True
            alarm_high = float(s.get("alarm_high", 999))
            state.current_value = alarm_high + random.uniform(0.2, 1.5)
            alarm = next((a for a in ALARM_TYPES if a["alarm_type"] == "pressure_high"), ALARM_TYPES[2])
            print(f"\n  🚨 シナリオ: 圧力スパイク — {s['tag_name']} ({state.equipment['name']})")
            print(f"     実測値: {state.current_value:.3f}{s['unit']}, アラーム閾値: {alarm_high}{s['unit']}")
            events.append(generate_process_alarm(state, alarm))

    # --- シナリオ 4: 品質逸脱（アクティブオーダーから）---
    if active_orders and random.random() < 0.10:
        order = random.choice(active_orders)
        product = products.get(order["product_id"], {})
        if product:
            spec_upper = float(product.get("spec_upper", 100))
            spec_lower = float(product.get("spec_lower", 0))
            offset = (spec_upper - spec_lower) * random.uniform(0.05, 0.15)
            measured = spec_upper + offset
            print(f"\n  🚨 シナリオ: 品質逸脱 — バッチ {order['batch_number']}")
            print(f"     製品: {product.get('name', 'Unknown')}, 測定値: {measured:.4f}, 規格上限: {spec_upper}")
            events.append(make_envelope("QualityInspectionEvent", "lims-system", {
                "process_order_id": order["process_order_id"],
                "batch_number": order["batch_number"],
                "product_id": order["product_id"],
                "inspection_item": "purity",
                "measured_value": round(measured, 4),
                "spec_lower": spec_lower,
                "spec_upper": spec_upper,
                "pass_fail": "fail",
                "lot_number": f"L-{datetime.utcnow().strftime('%Y')}-{random.randint(1000,9999):04d}",
            }))

    # --- シナリオ 5: バッチフェーズ遷移 ---
    if active_orders and random.random() < 0.15:
        order = random.choice(active_orders)
        phases = ["reaction", "distillation", "purification", "drying", "cooling", "mixing", "polymerization"]
        prev_phase = random.choice(phases)
        new_phase = random.choice([p for p in phases if p != prev_phase])
        print(f"\n  📋 フェーズ遷移 — バッチ {order['batch_number']}: {prev_phase} → {new_phase}")
        events.append(generate_batch_phase_transition(
            order, prev_phase, new_phase,
            random.randint(1, 5), order["production_line_id"]
        ))

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 35, Finished, Available, Finished, False)

## シミュレーション状態の初期化

In [ ]:
sim_states = []

for sensor in active_sensors:
    equip = equipment.get(sensor["equipment_id"])
    if not equip:
        continue
    line = production_lines.get(equip.get("production_line_id"))
    if not line:
        continue

    # このセンサーに関連するアクティブオーダーを探す
    order = None
    phase = None
    for o in active_orders:
        if o["production_line_id"] == line["production_line_id"]:
            order = o
            break

    state = PlantSimState(sensor, equip, line, order, phase)
    sim_states.append(state)

print(f"\nシミュレーション状態を初期化しました: {len(sim_states)} センサー")
for s in sim_states[:5]:
    print(f"  {s.sensor['tag_name']} ({s.sensor['measurement_type']}) "
          f"→ {s.equipment['name']} → {s.production_line['name']}")
if len(sim_states) > 5:
    print(f"  ... 他 {len(sim_states) - 5} センサー")

## イベント生成ループの実行

In [ ]:
SEPARATOR = "=" * 60
scenario_status = "ON" if SCENARIO_INJECTION_ENABLED else "OFF"
print(f"\n{SEPARATOR}")
print(f"イベント生成を開始します（{SIMULATION_DURATION_MINUTES}分間）")
print(f"アクティブセンサー: {len(sim_states)}")
print(f"計測間隔: {SENSOR_READING_INTERVAL_SECONDS}秒")
print(f"シナリオ注入: {scenario_status}")
print(f"送信先: Eventhouse ({KUSTO_DATABASE})")
print(f"{SEPARATOR}\n")

total_events = 0
start_time = time.time()
end_time = start_time + (SIMULATION_DURATION_MINUTES * 60)
tick = 0

try:
    while time.time() < end_time:
        tick += 1
        tick_events = []

        for state in sim_states:
            state.advance()

            # 1. センサー読み取りイベント（毎 tick）
            tick_events.append(generate_sensor_reading(state))

            # 2. アラーム閾値を超えた場合のアラームイベント
            if state.is_alarm:
                mtype = state.sensor.get("measurement_type", "")
                matching_alarms = [a for a in ALARM_TYPES if mtype in a["alarm_type"]]
                if matching_alarms:
                    tick_events.append(generate_process_alarm(state, random.choice(matching_alarms)))

            # 3. ランダムな品質検査イベント
            if state.order and random.random() < 0.03:
                product = products.get(state.order["product_id"])
                if product:
                    tick_events.append(generate_quality_inspection(state.order, product))

        # 4. デモシナリオの注入
        if SCENARIO_INJECTION_ENABLED:
            check_and_inject_scenarios(sim_states, tick_events)

        # バッファにイベントを追加
        for event in tick_events:
            if PRINT_EVENTS:
                body = event["body"]
                tag = body.get("tag_name", body.get("equipment_name", body.get("batch_number", "N/A")))
                print(f"  [{event['event_type']}] {tag}")
            buffer_event(event)

        total_events += len(tick_events)

        if tick % 6 == 0:
            flush_all_buffers()
            elapsed = time.time() - start_time
            print(f"\n--- Tick {tick} | {elapsed:.0f}秒経過 | {total_events} イベント生成済 ---")
            print(f"    取り込み数: {dict(_ingest_counts)}")

        time.sleep(SENSOR_READING_INTERVAL_SECONDS)

except KeyboardInterrupt:
    print("\n\n⚠ シミュレーションがユーザーにより停止されました。")

finally:
    flush_all_buffers()
    elapsed = time.time() - start_time
    print(f"\n{SEPARATOR}")
    print(f"シミュレーション完了")
    print(f"  実行時間: {elapsed:.0f}秒 ({elapsed/60:.1f}分)")
    print(f"  生成イベント合計: {total_events}")
    print(f"  テーブル別取り込み数:")
    for table, count in _ingest_counts.items():
        print(f"    {table:<35} {count:>6}")
    print(f"{SEPARATOR}")


In [ ]:
# Eventhouse テーブルのクエリ
print("Eventhouse 行数（Queued Ingestion の完了まで数分かかる場合があります）:\n")
for table in TABLE_COLUMNS:
    try:
        result = kusto_client.execute(KUSTO_DATABASE, f"{table} | count")
        for row in result.primary_results[0]:
            print(f"  {table:<35} {row[0]:>6} rows")
    except Exception as e:
        print(f"  {table:<35} ERROR: {e}")

# Step6.オントロジーの作成
ONTOROGY_NAME を書き換えて実行します。

In [42]:


ENTITY_CONFIG = {
    "production_lines":    {"entity_name": "ProductionLine",    "key_column": "production_line_id",    "display_column": "name"},
    "equipment":           {"entity_name": "Equipment",         "key_column": "equipment_id",          "display_column": "name"},
    "sensors":             {"entity_name": "Sensor",            "key_column": "sensor_id",             "display_column": "tag_name"},
    "products":            {"entity_name": "Product",           "key_column": "product_id",            "display_column": "name"},
    "process_orders":      {"entity_name": "ProcessOrder",      "key_column": "process_order_id",      "display_column": "batch_number"},
    "operation_phases":    {"entity_name": "OperationPhase",    "key_column": "operation_phase_id",    "display_column": "phase_name"},
    "process_deviations":  {"entity_name": "ProcessDeviation",  "key_column": "process_deviation_id",  "display_column": "deviation_type"},
    "quality_results":     {"entity_name": "QualityResult",     "key_column": "quality_result_id",     "display_column": "lot_number"},
    "failure_events":      {"entity_name": "FailureEvent",      "key_column": "failure_event_id",      "display_column": "failure_mode"},
    "root_causes":         {"entity_name": "RootCause",         "key_column": "root_cause_id",         "display_column": "cause_classification"},
}

RELATIONSHIPS = [
    {"name": "EquipmentOnLine",           "source_table": "equipment",           "source_entity": "Equipment",        "source_column": "equipment_id",          "target_entity": "ProductionLine",   "target_column": "production_line_id"},
    {"name": "SensorOnEquipment",         "source_table": "sensors",             "source_entity": "Sensor",           "source_column": "sensor_id",             "target_entity": "Equipment",        "target_column": "equipment_id"},
    {"name": "OrderOnLine",               "source_table": "process_orders",      "source_entity": "ProcessOrder",     "source_column": "process_order_id",      "target_entity": "ProductionLine",   "target_column": "production_line_id"},
    {"name": "PhaseInOrder",              "source_table": "operation_phases",    "source_entity": "OperationPhase",   "source_column": "operation_phase_id",    "target_entity": "ProcessOrder",     "target_column": "process_order_id"},
    {"name": "PhaseReadsSensor",          "source_table": "operation_phases",    "source_entity": "OperationPhase",   "source_column": "operation_phase_id",    "target_entity": "Sensor",           "target_column": "primary_sensor_id"},
    {"name": "DeviationFromSensor",       "source_table": "process_deviations", "source_entity": "ProcessDeviation", "source_column": "process_deviation_id",  "target_entity": "Sensor",           "target_column": "sensor_id"},
    {"name": "DeviationInPhase",          "source_table": "process_deviations", "source_entity": "ProcessDeviation", "source_column": "process_deviation_id",  "target_entity": "OperationPhase",   "target_column": "operation_phase_id"},
    {"name": "DeviationAffectsQuality",   "source_table": "quality_results",    "source_entity": "QualityResult",    "source_column": "quality_result_id",     "target_entity": "ProcessDeviation", "target_column": "process_deviation_id"},
    {"name": "DeviationEscalatesFailure", "source_table": "failure_events",     "source_entity": "FailureEvent",     "source_column": "failure_event_id",      "target_entity": "ProcessDeviation", "target_column": "process_deviation_id"},
    {"name": "FailureCausedByRootCause",  "source_table": "root_causes",        "source_entity": "RootCause",        "source_column": "root_cause_id",         "target_entity": "FailureEvent",     "target_column": "failure_event_id"},
    {"name": "RootCauseEquipment",        "source_table": "root_causes",        "source_entity": "RootCause",        "source_column": "root_cause_id",         "target_entity": "Equipment",        "target_column": "equipment_id"},
    {"name": "RootCausePhase",            "source_table": "root_causes",        "source_entity": "RootCause",        "source_column": "root_cause_id",         "target_entity": "OperationPhase",   "target_column": "operation_phase_id"},
    {"name": "OrderProducesProduct",      "source_table": "process_orders",     "source_entity": "ProcessOrder",     "source_column": "process_order_id",      "target_entity": "Product",          "target_column": "product_id"},
    {"name": "QualityForProduct",         "source_table": "quality_results",    "source_entity": "QualityResult",    "source_column": "quality_result_id",     "target_entity": "Product",          "target_column": "product_id"},
]

# Spark simpleString() → Ontology valueType
SPARK_TO_ONTOLOGY = {
    "string":  "String",
    "boolean": "Boolean",
    "date":    "DateTime",
    "timestamp": "DateTime",
    "int":     "BigInt",
    "integer": "BigInt",
    "short":   "BigInt",
    "byte":    "BigInt",
    "long":    "BigInt",
    "bigint":  "BigInt",
    "float":   "Double",
    "double":  "Double",
}

StatementMeta(, ac09c340-1739-489b-adc1-28c3885f0364, 44, Finished, Available, Finished, False)

In [ ]:
table_schemas = {}

for table_name, cfg in ENTITY_CONFIG.items():
    schema = spark.table(f"`{WORKSPACE_NAME}`.{LAKEHOUSE_NAME}.{table_name}").schema

    columns = []
    for field in schema.fields:
        type_str = field.dataType.simpleString()
        ontology_type = SPARK_TO_ONTOLOGY.get(type_str)
        if ontology_type:
            columns.append({"name": field.name, "valueType": ontology_type})
        else:
            print(f"  ⚠️  Skipping {table_name}.{field.name} (type: {type_str})")

    table_schemas[table_name] = columns
    print(f"  ✅ {table_name}: {len(columns)} compatible columns")

print(f"\n📊 Total: {sum(len(c) for c in table_schemas.values())} columns across {len(table_schemas)} tables")

In [ ]:
import json, base64, uuid, random

random.seed(42)
_used_ids = set()


def generate_id():
    """Generate a unique positive 64-bit integer ID (as a string)."""
    while True:
        id_val = random.randint(10**12, 10**15)
        if id_val not in _used_ids:
            _used_ids.add(id_val)
            return str(id_val)


def to_base64(obj):
    return base64.b64encode(json.dumps(obj).encode("utf-8")).decode("utf-8")


# ── Tracking maps ────────────────────────────────────────────────────────────
entity_type_ids  = {}   # entity_name → entity_type_id
property_ids     = {}   # (entity_name, col_name) → property_id
key_property_ids = {}   # entity_name → key property_id

# ── Start with the required empty definition part ────────────────────────────
parts = [
    {"path": "definition.json", "payload": to_base64({}), "payloadType": "InlineBase64"},
]

# ── Entity types + data bindings ─────────────────────────────────────────────
for table_name, cfg in ENTITY_CONFIG.items():
    entity_name = cfg["entity_name"]
    key_column  = cfg["key_column"]
    columns     = table_schemas[table_name]

    entity_type_id = generate_id()
    entity_type_ids[entity_name] = entity_type_id

    properties = []
    for col in columns:
        prop_id = generate_id()
        property_ids[(entity_name, col["name"])] = prop_id
        properties.append({
            "id": prop_id,
            "name": col["name"],
            "redefines": None,
            "baseTypeNamespaceType": None,
            "valueType": col["valueType"],
        })

    key_prop_id = property_ids[(entity_name, key_column)]
    key_property_ids[entity_name] = key_prop_id

    # Entity type definition
    entity_def = {
        "id": entity_type_id,
        "namespace": "usertypes",
        "baseEntityTypeId": None,
        "name": entity_name,
        "entityIdParts": [key_prop_id],
        "displayNamePropertyId": property_ids[(entity_name, cfg.get("display_column", key_column))],
        "namespaceType": "Custom",
        "visibility": "Visible",
        "properties": properties,
        "timeseriesProperties": [],
    }
    parts.append({
        "path": f"EntityTypes/{entity_type_id}/definition.json",
        "payload": to_base64(entity_def),
        "payloadType": "InlineBase64",
    })

    # Data binding (NonTimeSeries → lakehouse table)
    binding_id = str(uuid.uuid4())
    data_binding = {
        "id": binding_id,
        "dataBindingConfiguration": {
            "dataBindingType": "NonTimeSeries",
            "propertyBindings": [
                {"sourceColumnName": col["name"], "targetPropertyId": property_ids[(entity_name, col["name"])]}
                for col in columns
            ],
            "sourceTableProperties": {
                "sourceType": "LakehouseTable",
                "workspaceId": WORKSPACE_ID,
                "itemId": LAKEHOUSE_ID,
                "sourceTableName": table_name
            },
        },
    }
    parts.append({
        "path": f"EntityTypes/{entity_type_id}/DataBindings/{binding_id}.json",
        "payload": to_base64(data_binding),
        "payloadType": "InlineBase64",
    })
    print(f"  ✅ {entity_name}  ({len(properties)} properties, key={key_column})")

# ── Relationship types + contextualizations ──────────────────────────────────
for rel in RELATIONSHIPS:
    rel_id = generate_id()
    source_entity = rel["source_entity"]
    target_entity = rel["target_entity"]

    rel_def = {
        "namespace": "usertypes",
        "id": rel_id,
        "name": rel["name"],
        "namespaceType": "Custom",
        "source": {"entityTypeId": entity_type_ids[source_entity]},
        "target": {"entityTypeId": entity_type_ids[target_entity]},
    }
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/definition.json",
        "payload": to_base64(rel_def),
        "payloadType": "InlineBase64",
    })

    # Contextualization — binds columns in the source table to entity keys
    ctx_id = str(uuid.uuid4())
    contextualization = {
        "id": ctx_id,
        "dataBindingTable": {
            "workspaceId": WORKSPACE_ID,
            "itemId": LAKEHOUSE_ID,
            "sourceTableName": rel["source_table"],
            "sourceType": "LakehouseTable",
        },
        "sourceKeyRefBindings": [
            {"sourceColumnName": rel["source_column"], "targetPropertyId": key_property_ids[source_entity]}
        ],
        "targetKeyRefBindings": [
            {"sourceColumnName": rel["target_column"], "targetPropertyId": key_property_ids[target_entity]}
        ],
    }
    parts.append({
        "path": f"RelationshipTypes/{rel_id}/Contextualizations/{ctx_id}.json",
        "payload": to_base64(contextualization),
        "payloadType": "InlineBase64",
    })
    print(f"  ✅ {rel['name']}  ({source_entity} → {target_entity})")

print(f"\n📦 Total definition parts: {len(parts)}")

In [ ]:
import time

# ── Delete existing ontology if present ──────────────────────────────────────
resp = client.get(f"v1/workspaces/{WORKSPACE_ID}/ontologies")
resp.raise_for_status()
existing = next(
    (o for o in resp.json().get("value", []) if o["displayName"] == ONTOLOGY_NAME),
    None,
)
if existing:
    print(f"🗑️  Deleting existing ontology '{ONTOLOGY_NAME}' (id={existing['id']})...")
    del_resp = client.delete(f"v1/workspaces/{WORKSPACE_ID}/ontologies/{existing['id']}")
    del_resp.raise_for_status()
    print(f"   ⏳ Deleted. Waiting 30 seconds for cleanup...")
    time.sleep(30)
    print(f"   ✅ 30 second wait completed...proceeding with new ontology creation.") 

# ── Create ontology with full definition ─────────────────────────────────────
payload = {
    "displayName": ONTOLOGY_NAME,
    "description": "Chemical manufacturing domain ontology — entities and relationships for process operations.",
    "definition": {"parts": parts},
}

print(f"⏳ Creating ontology '{ONTOLOGY_NAME}'...")
resp = client.post(f"v1/workspaces/{WORKSPACE_ID}/ontologies", json=payload)

if resp.status_code == 201:
    result = resp.json()
    print(f"✅ Ontology created!  ID: {result['id']}")

elif resp.status_code == 202:
    operation_id = resp.headers.get("x-ms-operation-id")
    retry_after  = int(resp.headers.get("Retry-After", "30"))
    print(f"   Operation {operation_id} — polling every {retry_after}s...")

    while True:
        time.sleep(retry_after)
        poll = client.get(f"v1/operations/{operation_id}")
        poll.raise_for_status()
        status = poll.json().get("status")
        print(f"   Status: {status}")

        if status == "Succeeded":
            result_resp = client.get(f"v1/operations/{operation_id}/result")
            if result_resp.status_code == 200:
                print(f"✅ Ontology created!  ID: {result_resp.json().get('id', 'N/A')}")
            else:
                print(f"✅ Ontology creation succeeded.")
            break
        elif status in ("Failed", "Cancelled"):
            print(f"❌ Ontology creation {status.lower()}.")
            print(f"   {poll.json()}")
            break
else:
    print(f"❌ HTTP {resp.status_code}: {resp.text}")
    resp.raise_for_status()